In [ ]:
import sys
print(sys.executable)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
import numpy as np
import optuna
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.integration.keras import TuneReportCallback
from ray.tune import TuneConfig
from ray.tune import Tuner, TuneConfig
from ray.air.config import RunConfig
from ray.tune.progress_reporter import CLIReporter
import time,psutil
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from optuna.integration import TFKerasPruningCallback



In [ ]:

# Parametry
N = 10000
rng = np.random.default_rng(42)

# Klasy: 0 - karpiowy, 1 - liniowy, 2 - pstrągowy
classes = rng.choice([0, 1, 2], size=N, p=[0.4, 0.35, 0.25])

# Cechy środowiskowe i biologiczne
temperatura = np.select([classes==0, classes==1, classes==2],
                        [rng.normal(20, 3, N), rng.normal(18, 3, N), rng.normal(15, 3, N)])
ph = np.select([classes==0, classes==1, classes==2],
               [rng.normal(7.3, 0.3, N), rng.normal(7.1, 0.3, N), rng.normal(6.9, 0.3, N)])
tlen = np.select([classes==0, classes==1, classes==2],
                 [rng.normal(6.8, 0.6, N), rng.normal(7.2, 0.6, N), rng.normal(7.8, 0.6, N)])
azotany = np.select([classes==0, classes==1, classes==2],
                    [rng.normal(2.8, 0.9, N), rng.normal(2.6, 0.9, N), rng.normal(2.0, 0.9, N)])
fosforany = np.select([classes==0, classes==1, classes==2],
                      [rng.normal(0.55, 0.2, N), rng.normal(0.5, 0.2, N), rng.normal(0.45, 0.2, N)])
przezroczystosc = np.select([classes==0, classes==1, classes==2],
                            [rng.normal(1.5, 0.4, N), rng.normal(1.6, 0.4, N), rng.normal(1.8, 0.4, N)])
gleba = np.select([classes==0, classes==1, classes==2],
                  [rng.normal(5.0, 1.2, N), rng.normal(4.8, 1.2, N), rng.normal(5.3, 1.2, N)])
powierzchnia = np.select([classes==0, classes==1, classes==2],
                         [rng.normal(2.2, 0.5, N), rng.normal(2.0, 0.5, N), rng.normal(2.4, 0.5, N)])
glebokosc = np.select([classes==0, classes==1, classes==2],
                      [rng.normal(1.8, 0.4, N), rng.normal(2.0, 0.4, N), rng.normal(2.3, 0.4, N)])
roslinnosc = np.select([classes==0, classes==1, classes==2],
                       [rng.normal(0.7, 0.2, N), rng.normal(0.8, 0.2, N), rng.normal(0.6, 0.2, N)])
przeplyw = np.select([classes==0, classes==1, classes==2],
                     [rng.normal(0.35, 0.1, N), rng.normal(0.4, 0.1, N), rng.normal(0.5, 0.1, N)])
srednia_waga_ryb = np.select([classes==0, classes==1, classes==2],
                             [rng.normal(1.0, 0.3, N), rng.normal(0.8, 0.3, N), rng.normal(0.6, 0.3, N)])

# DataFrame
df = pd.DataFrame({
    "temperatura_wody": temperatura,
    "pH": ph,
    "tlen": tlen,
    "azotany": azotany,
    "fosforany": fosforany,
    "przezroczystość": przezroczystosc,
    "gleba": gleba,
    "powierzchnia_stawu": powierzchnia,
    "głębokość": glebokosc,
    "roślinność": roslinnosc,
    "przepływ": przeplyw,
    "średnia_waga_ryb": srednia_waga_ryb,
    "typ_stawu": classes
})

df.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sample_df = df.sample(5000, random_state=42)

palette_map = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}

plt.figure(figsize=(8, 6))
ax = sns.scatterplot(
    data=sample_df,
    x="temperatura_wody",
    y="tlen",
    hue="typ_stawu",
    palette=palette_map,
    alpha=0.7
)
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=["karpiowy (0)", "linowy (1)", "pstrągowy (2)"], title="Typ stawu")

plt.title("Rozkład danych dla przykładowych cech")
plt.xlabel("Temperatura wody [°C]")
plt.ylabel("Zawartość tlenu [mg/L]")
plt.show()

In [ ]:
#zaokrąglenie dla czytelności
df = df.round(3)
df.head()

In [ ]:
print(df.info())


In [ ]:
print(df.describe().T)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Podział 70/30: trening/test
X = df.drop(columns=["typ_stawu"])
y = df["typ_stawu"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# walidacja wydzielona z treningu  20% z treningu)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

print("Treningowe:", X_train.shape)
print("Walidacyjne:", X_val.shape)
print("Testowe:", X_test.shape)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)
x_val_scaled   = scaler.transform(X_val)
x_test_scaled  = scaler.transform(X_test)


## Model Bazowy Autoenkoder

In [ ]:
input_dim = x_train_scaled.shape[1]
num_classes = len(np.unique(y_train))
encoding_dim = 6  

def build_autoencoder():
    input_layer = layers.Input(shape=(input_dim,))
    
    # ENCODER
    encoded = layers.Dense(32, activation='relu')(input_layer)
    encoded = layers.Dense(16, activation='relu')(encoded)
    bottleneck = layers.Dense(encoding_dim, activation='relu', name="bottleneck")(encoded)
    
    # DECODER
    decoded = layers.Dense(16, activation='relu')(bottleneck)
    decoded = layers.Dense(32, activation='relu')(decoded)
    output_layer = layers.Dense(input_dim, activation='linear', name='reconstruction')(decoded)
    
    # KLASYFIKATOR
    clas =  layers.Dense(16, activation='relu')(bottleneck)
    class_out = layers.Dense(num_classes, activation='softmax', name='class_output')(clas)
    
    # MODEL
    autoencoder = keras.Model(inputs=input_layer, outputs=[class_out, output_layer])
    optimizer = keras.optimizers.Adam(learning_rate=0.005)

    autoencoder.compile(
        optimizer = optimizer,


        loss={ 
            'class_output': 'sparse_categorical_crossentropy', 
            'reconstruction': 'mse'
        },
        loss_weights={
            'class_output': 1.0,
            'reconstruction': 0.5 
        },
        metrics={
            'class_output': 'accuracy'
        }
    )
    
    return autoencoder

#Inicjalizacja modelu
autoencoder = build_autoencoder()

start = time.perf_counter()
process = psutil.Process()

history = autoencoder.fit(
    x_train_scaled, 
    {'class_output':y_train #etykiety stawu
    , 'reconstruction':x_train_scaled},
    validation_data=(x_val_scaled, {'class_output': y_val, 'reconstruction': x_val_scaled}),
    epochs=50,
    batch_size=64,
    shuffle=True,
    verbose=1
)

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024*1024)
print(f"Czas uczenia: {elapsed:.2f}s")
print(f"Zużycie pamięci: {mem_used:.2f} MB")


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history['class_output_accuracy'], label='Train Accuracy')
plt.plot(history.history['val_class_output_accuracy'], label='Validation Accuracy')
plt.title('Classification Accuracy (Train vs Validation)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
_, x_test_pred = autoencoder.predict(x_test_scaled)
# MSE dla każdej próbki testowej
recon_errors = np.mean((x_test_scaled - x_test_pred) ** 2, axis=1)
recon_errors[:10]


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(recon_errors, bins=40, edgecolor='black')
plt.xlabel("Reconstruction error (MSE)")
plt.ylabel("Liczba próbek")
plt.grid(True, alpha=0.3)
plt.show()


## Keras Tuner : Random Search

In [ ]:
input_dim = x_train_scaled.shape[1]
num_classes = len(np.unique(y_train))
def build_autoencoder_model(hp):
    #Przedział wyborów HP
    hp_units_1 = hp.Int('units_1', min_value=16, max_value=128, step=16)
    hp_units_2 = hp.Int('units_2', min_value=8, max_value=64, step=8)
    hp_encoded = hp.Int('encoding_dim', min_value=4, max_value=12, step=2)
    hp_learning_rate = hp.Choice('learning_rate', values=[0.001, 0.003, 0.005, 0.01])
    input_layer = layers.Input(shape=(input_dim,))
    hp_class_units = hp.Int('class_units', min_value=8, max_value=64, step=8)


    #Encoder
    encoded = layers.Dense(hp_units_1, activation='relu')(input_layer)
    encoded = layers.Dense(hp_units_2, activation='relu')(encoded)
    bottleneck = layers.Dense(hp_encoded, activation='relu', name='bottleneck')(encoded)

    #Decoder
    decoded = layers.Dense(hp_units_2, activation='relu')(bottleneck)
    decoded = layers.Dense(hp_units_1, activation='relu')(decoded)
    output_layer = layers.Dense(input_dim, activation='linear', name='reconstruction')(decoded)

    #Klasyfikator
    clas = layers.Dense(hp_class_units, activation='relu')(bottleneck)
    class_out = layers.Dense(num_classes, activation='softmax', name='class_output')(clas)

    #Model
    autoencoder = keras.Model(inputs=input_layer, outputs=[class_out, output_layer])
    optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)
    
    autoencoder.compile(
        optimizer=optimizer,
        loss={
            'class_output': 'sparse_categorical_crossentropy',
            'reconstruction': 'mse'
        },
        loss_weights={
            'class_output':1.0, 
            'reconstruction':0.5 
        },
        metrics={
            'class_output': SparseCategoricalAccuracy(name='accuracy')
        }
    )

    return autoencoder


callback = keras.callbacks.EarlyStopping(
    monitor='val_class_output_accuracy',  
    patience=10,
    mode='max',
    restore_best_weights=True
)

start = time.perf_counter()
process = psutil.Process()

tuner = kt.RandomSearch(
    build_autoencoder_model,
    objective=kt.Objective('val_class_output_accuracy', direction='max'),
    max_trials=20,
    executions_per_trial = 1,
    directory="C:\\Users\\ASUS ZENBOOK\\Desktop",
    project_name = "random_search_autoencoder_class10"
)

tuner.search(
    x_train_scaled, 
    {'class_output': y_train, 'reconstruction': x_train_scaled},
    validation_data=(
        x_val_scaled,
        {'class_output': y_val, 'reconstruction': x_val_scaled}
    ),
    epochs=50,
    callbacks=[callback],
    batch_size=64,
    verbose=1
)

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024*1024)
print(f"\nCzas wyszukiwania: {round(elapsed,2)}")
print(f"\nZużycie pamięci: {round(mem_used,2)}")

#najlepsze hiperparametry
best_hp_rs = tuner.get_best_hyperparameters(1)[0]
best_model_rs = tuner.hypermodel.build(best_hp_rs)

print(f"""
Najlepsze hiperparametry:
- units_1: {best_hp_rs.get('units_1')}
- units_2: {best_hp_rs.get('units_2')}
- encoding_dim: {best_hp_rs.get('encoding_dim')}
- learning_rate: {best_hp_rs.get('learning_rate')}
- class units : {best_hp_rs.get('class_units')}
""")

history_best_rs = best_model_rs.fit(
    x_train_scaled,
    {'class_output': y_train, 'reconstruction': x_train_scaled},
    validation_data=(
        x_val_scaled,
        {'class_output': y_val, 'reconstruction': x_val_scaled}
    ),
    epochs=50,
    batch_size=64,
    callbacks=[callback],
    verbose=1
)

eval_results_rs = best_model_rs.evaluate(
    x_test_scaled,
    {'class_output': y_test, 'reconstruction': x_test_scaled},
    verbose=0,
    return_dict=True
)

print("\nWyniki na zbiorze testowym (surowe):")
print(eval_results_rs)

print("\nŁadnie sformatowane:")
print(f"total_loss:            {eval_results_rs['loss']:.4f}")
print(f"class_output_loss:     {eval_results_rs['class_output_loss']:.4f}")
print(f"reconstruction_loss:   {eval_results_rs['reconstruction_loss']:.4f}")
print(f"class_output_accuracy: {eval_results_rs['class_output_accuracy']:.4f}")



In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history_best_rs.history['class_output_accuracy'], label='Train Accuracy')
plt.plot(history_best_rs.history['val_class_output_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Predykcja rekonstrukcji (wyjście drugie)
class_pred, x_test_pred = best_model_rs.predict(x_test_scaled)

# Błąd rekonstrukcji MSE dla każdej próbki
recon_errors_rs = np.mean((x_test_scaled - x_test_pred)**2, axis=1)
print(recon_errors_rs[:10])

plt.figure(figsize=(8,5))
plt.hist(recon_errors_rs, bins=40, edgecolor='black')
plt.xlabel("Reconstruction error (MSE)")
plt.ylabel("Liczba próbek")
plt.grid(True, alpha=0.3)
plt.show()


## Hyperband

In [ ]:
input_dim = x_train_scaled.shape[1]
num_classes = len(np.unique(y_train))

def build_autoencoder_model(hp):
    #Przedział wyborów HP
    hp_units_1 = hp.Int('units_1', min_value=16, max_value=128, step=16)
    hp_units_2 = hp.Int('units_2', min_value=8, max_value=64, step=8)
    hp_encoded = hp.Int('encoding_dim', min_value=4, max_value=12, step=2)
    hp_learning_rate = hp.Choice('learning_rate', values=[0.001, 0.003, 0.005, 0.01])
    input_layer = layers.Input(shape=(input_dim,))
    hp_class_units = hp.Int('class_units', min_value=8, max_value=64, step=8)

    #Encoder
    encoded = layers.Dense(hp_units_1, activation='relu')(input_layer)
    encoded = layers.Dense(hp_units_2, activation='relu')(encoded)
    bottleneck = layers.Dense(hp_encoded, activation='relu', name='bottleneck')(encoded)

    #Decoder
    decoded = layers.Dense(hp_units_2, activation='relu')(bottleneck)
    decoded = layers.Dense(hp_units_1, activation='relu')(decoded)
    output_layer = layers.Dense(input_dim, activation='linear', name='reconstruction')(decoded)

    #Klasyfikator
    clas = layers.Dense(hp_class_units, activation='relu')(bottleneck)
    class_out = layers.Dense(num_classes, activation='softmax', name='class_output')(clas)

    #Model
    autoencoder = keras.Model(inputs=input_layer, outputs=[class_out, output_layer])
    optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)

    autoencoder.compile(
        optimizer=optimizer,
        loss={
            'class_output': 'sparse_categorical_crossentropy',
            'reconstruction': 'mse'
        },
        loss_weights={
            'class_output': 1.0,
            'reconstruction': 0.5
        },
        metrics={
            'class_output': SparseCategoricalAccuracy(name='accuracy')
        }
    )

    return autoencoder


callback = keras.callbacks.EarlyStopping(
    monitor='val_class_output_accuracy',
    patience=10,
    mode='max',
    restore_best_weights=True
)

start = time.perf_counter()
process = psutil.Process()

tuner = kt.Hyperband(
    hypermodel=build_autoencoder_model,
    objective=kt.Objective("val_class_output_accuracy", direction="max"),
    max_epochs=50,
    factor=3,
    directory="C:\\Users\\ASUS ZENBOOK\\Desktop",
    project_name="hyperband_autoencoder_class3.3"
)

tuner.search(
    x_train_scaled,
    {'class_output': y_train, 'reconstruction': x_train_scaled},
    validation_data=(
        x_val_scaled,
        {'class_output': y_val, 'reconstruction': x_val_scaled}
    ),
    epochs=50,
    callbacks=[callback],
    batch_size=64,
    verbose=1
)

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024 * 1024)
print(f"Czas szukania: {round(elapsed, 2)}s")
print(f"Zużycie pamięci: {round(mem_used, 2)}MB")

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model_hyperband = tuner.hypermodel.build(best_hps)

print(f"""
Najlepsze hiperparametry:
- units_1: {best_hps.get('units_1')}
- units_2: {best_hps.get('units_2')}
- encoding_dim: {best_hps.get('encoding_dim')}
- learning_rate: {best_hps.get('learning_rate')}
- calss_units: {best_hps.get('class_units')}
""")

history_best_hyperbadnd = best_model_hyperband.fit(
    x_train_scaled,
    {'class_output': y_train, 'reconstruction': x_train_scaled},
    validation_data=(
        x_val_scaled,
        {'class_output': y_val, 'reconstruction': x_val_scaled}
    ),
    epochs=50,
    batch_size=64,
    callbacks=[callback],
    verbose=1
)

eval_results_hyperband = best_model_hyperband.evaluate(
    x_test_scaled,
    {'class_output': y_test, 'reconstruction': x_test_scaled},
    verbose=0,
    return_dict=True
)

print("Wyniki================================================")
print(f"total_loss:{eval_results_hyperband['loss']:.4f}")
print(f"class_output_loss:{eval_results_hyperband['class_output_loss']:.4f}")
print(f"reconstruction_loss:{eval_results_hyperband['reconstruction_loss']:.4f}")
print(f"class_output_accuracy:{eval_results_hyperband['class_output_accuracy']:.4f}")


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history_best_hyperbadnd.history['class_output_accuracy'], label='Train Accuracy')
plt.plot(history_best_hyperbadnd.history['val_class_output_accuracy'], label='Validation Accuarcy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Predykcja rekonstrukcji (wyjście drugie)
class_pred, x_test_pred = best_model_hyperband.predict(x_test_scaled)

# Błąd rekonstrukcji MSE dla każdej próbki
recon_errors_hyperband = np.mean((x_test_scaled - x_test_pred)**2, axis=1)
print(recon_errors_hyperband[:10])

plt.figure(figsize=(8,5))
plt.hist(recon_errors_hyperband, bins=40, edgecolor='black')
plt.xlabel("Reconstruction error (MSE)")
plt.ylabel("Liczba próbek")
plt.grid(True, alpha=0.3)
plt.show()


## Optuna pruning

In [ ]:
input_dim = x_train_scaled.shape[1]
num_classes = len(np.unique(y_train))

def build_autoencoder(units_1, units_2, encoding_dim, class_units, learning_rate):
    # WEJSCIE
    inp = layers.Input(shape=(input_dim,))

    # ENCODER
    x = layers.Dense(units_1, activation='relu')(inp)
    x = layers.Dense(units_2, activation='relu')(x)
    bottleneck = layers.Dense(encoding_dim, activation='relu', name="bottleneck")(x)

    # DECODER
    d = layers.Dense(units_2, activation='relu')(bottleneck)
    d = layers.Dense(units_1, activation='relu')(d)
    reconstruction = layers.Dense(input_dim, activation='linear', name="reconstruction")(d)

    # KLASYFIKATOR
    c = layers.Dense(class_units, activation='relu')(bottleneck)
    class_out = layers.Dense(num_classes, activation='softmax', name="class_output")(c)

    model = keras.Model(inputs=inp, outputs=[class_out, reconstruction])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss={
            "class_output": "sparse_categorical_crossentropy",
            "reconstruction": "mse"
        },
        loss_weights={
            "class_output": 1.0,
            "reconstruction": 0.5
        },
        metrics={
            "class_output": SparseCategoricalAccuracy(name="accuracy")
        }
    )
    return model


def objective(trial):
    # HIPERPARAMETRY Optuny
    units_1 = trial.suggest_int("units_1", 16, 128, step=16)
    units_2 = trial.suggest_int("units_2", 8, 64, step=8)
    encoding_dim = trial.suggest_int("encoding_dim", 4, 12, step=2)
    class_units = trial.suggest_int("class_units", 8, 64, step=8)
    learning_rate = trial.suggest_categorical("learning_rate", [0.001, 0.003, 0.005, 0.01])

    model = build_autoencoder(units_1, units_2, encoding_dim, class_units, learning_rate)

    es = keras.callbacks.EarlyStopping(
        monitor="val_class_output_accuracy",
        patience=10,
        mode="max",
        restore_best_weights=True
    )

    pruning = TFKerasPruningCallback(
        trial,
        monitor="val_class_output_accuracy"
    )

    history = model.fit(
        x_train_scaled,
        {"class_output": y_train, "reconstruction": x_train_scaled},
        validation_data=(
            x_val_scaled,
            {"class_output": y_val, "reconstruction": x_val_scaled}
        ),
        epochs=50,
        batch_size=64,
        verbose=0,
        callbacks=[es, pruning]
    )

    return max(history.history["val_class_output_accuracy"])


start = time.perf_counter()
process = psutil.Process()

study = optuna.create_study(
    direction="maximize",
    study_name="supervised_autoencoder_optuna9"
)

study.optimize(objective, n_trials=20)

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024 * 1024)

print(f"\nCzas szukania: {elapsed:.2f} s")
print(f"Zużycie pamięci: {mem_used:.2f} MB\n")

print("Najlepsze hiperparametry:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("\nNajlepsze val accuracy:", study.best_value)

best_params = study.best_params

best_model_optuna_pruning = build_autoencoder(
    units_1=best_params["units_1"],
    units_2=best_params["units_2"],
    encoding_dim=best_params["encoding_dim"],
    class_units=best_params["class_units"],
    learning_rate=best_params["learning_rate"]
)

es_final = keras.callbacks.EarlyStopping(
    monitor="val_class_output_accuracy",
    patience=10,
    mode="max",
    restore_best_weights=True
)

history_final_optuna_pruning = best_model_optuna_pruning.fit(
    x_train_scaled,
    {"class_output": y_train, "reconstruction": x_train_scaled},
    validation_data=(
        x_val_scaled,
        {"class_output": y_val, "reconstruction": x_val_scaled}
    ),
    epochs=50,
    batch_size=64,
    verbose=1,
    callbacks=[es_final]
)

# EWALUACJA na zbiorze testowym
results_optuna_pruning = best_model_optuna_pruning.evaluate(
    x_test_scaled,
    {"class_output": y_test, "reconstruction": x_test_scaled},
    return_dict=True
)

print("\nWyniki na zbiorze testowym:")
for k, v in results_optuna_pruning.items():
    print(f"{k}: {v:.4f}")


In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history_final_optuna_pruning.history["class_output_accuracy"],label ='Train Accuracy')
plt.plot(history_final_optuna_pruning.history["val_class_output_accuracy"], label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Predykcja rekonstrukcji (wyjście drugie) z najlepszego modelu Optuny
class_pred_optuna, x_test_pred_optuna = best_model_optuna_pruning.predict(x_test_scaled)

# Błąd rekonstrukcji MSE dla każdej próbki
recon_errors_optuna_pruning = np.mean((x_test_scaled - x_test_pred_optuna) ** 2, axis=1)
print(recon_errors_optuna_pruning[:10])

plt.figure(figsize=(8, 5))
plt.hist(recon_errors_optuna_pruning, bins=40, edgecolor='black')
plt.xlabel("Reconstruction error (MSE)")
plt.ylabel("Liczba próbek")
plt.grid(True, alpha=0.3)
plt.show()


## Optuna bez pruning

In [ ]:
input_dim = x_train_scaled.shape[1]
num_classes = len(np.unique(y_train))

# FUNKCJA BUDUJĄCA MODEL (czysta)
def build_autoencoder(units_1, units_2, encoding_dim, class_units, learning_rate):

    inp = layers.Input(shape=(input_dim,))

    # ENCODER
    x = layers.Dense(units_1, activation='relu')(inp)
    x = layers.Dense(units_2, activation='relu')(x)
    bottleneck = layers.Dense(encoding_dim, activation='relu', name="bottleneck")(x)

    # DECODER
    d = layers.Dense(units_2, activation='relu')(bottleneck)
    d = layers.Dense(units_1, activation='relu')(d)
    reconstruction = layers.Dense(input_dim, activation='linear', name="reconstruction")(d)

    # KLASYFIKATOR
    c = layers.Dense(class_units, activation='relu')(bottleneck)
    class_out = layers.Dense(num_classes, activation='softmax', name="class_output")(c)

    # MODEL
    model = keras.Model(inputs=inp, outputs=[class_out, reconstruction])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss={
            "class_output": "sparse_categorical_crossentropy",
            "reconstruction": "mse"
        },
        loss_weights={
            "class_output": 1.0,
            "reconstruction": 0.5
        },
        metrics={
            "class_output": SparseCategoricalAccuracy(name="accuracy")
        }
    )

    return model


# FUNKCJA CELU DO OPTUNY (BEZ PRUNINGU)
def objective(trial):
    units_1 = trial.suggest_int("units_1", 16, 128, step=16)
    units_2 = trial.suggest_int("units_2", 8, 64, step=8)
    encoding_dim = trial.suggest_int("encoding_dim", 4, 12, step=2)
    class_units = trial.suggest_int("class_units", 8, 64, step=8)
    learning_rate = trial.suggest_categorical("learning_rate", [0.001, 0.003, 0.005, 0.01])

    model = build_autoencoder(units_1, units_2, encoding_dim, class_units, learning_rate)

    es = keras.callbacks.EarlyStopping(
        monitor="val_class_output_accuracy",
        patience=10,
        mode="max",
        restore_best_weights=True
    )

    # TRENING (walidacja na zbiorze walidacyjnym, nie testowym)
    history = model.fit(
        x_train_scaled,
        {"class_output": y_train, "reconstruction": x_train_scaled},
        validation_data=(
            x_val_scaled,
            {"class_output": y_val, "reconstruction": x_val_scaled}
        ),
        epochs=50,
        batch_size=64,
        verbose=0,
        callbacks=[es]
    )

    return max(history.history["val_class_output_accuracy"])


start = time.perf_counter()
process = psutil.Process()

study = optuna.create_study(
    direction="maximize",
    study_name="supervised_autoencoder_optuna_bez4"
)

study.optimize(objective, n_trials=20)

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024 * 1024)

print(f"\nCzas szukania: {elapsed:.2f} s")
print(f"Zużycie pamięci: {mem_used:.2f} MB\n")

print("Najlepsze hiperparametry:")
for k, v in study.best_params.items():
    print(f"{k}: {v}")

print("\nNajlepsze val accuracy:", study.best_value)

best_params = study.best_params

best_model_optuna_bez = build_autoencoder(
    units_1=best_params["units_1"],
    units_2=best_params["units_2"],
    encoding_dim=best_params["encoding_dim"],
    class_units=best_params["class_units"],
    learning_rate=best_params["learning_rate"]
)

es_final = keras.callbacks.EarlyStopping(
    monitor="val_class_output_accuracy",
    patience=10,
    mode="max",
    restore_best_weights=True
)

history_final_optuna_bez_purning = best_model_optuna_bez.fit(
    x_train_scaled,
    {"class_output": y_train, "reconstruction": x_train_scaled},
    validation_data=(
        x_val_scaled,
        {"class_output": y_val, "reconstruction": x_val_scaled}
    ),
    epochs=50,
    batch_size=64,
    verbose=1,
    callbacks=[es_final]
)

# EWALUACJA na zbiorze testowym (tylko na końcu)
results_optuna_bez_purning = best_model_optuna_bez.evaluate(
    x_test_scaled,
    {"class_output": y_test, "reconstruction": x_test_scaled},
    return_dict=True
)

print("\nWyniki na zbiorze testowym:")
for k, v in results_optuna_bez_purning.items():
    print(f"{k}: {v:.4f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_final_optuna_bez_purning.history['class_output_accuracy'], label='Train Accuracy')
plt.plot(history_final_optuna_bez_purning.history['val_class_output_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Predykcja rekonstrukcji (wyjście drugie) z najlepszego modelu Optuny
class_pred_optuna, x_test_pred_optuna = best_model_optuna_bez.predict(x_test_scaled)

# Błąd rekonstrukcji MSE dla każdej próbki
recon_errors_optuna_bez = np.mean((x_test_scaled - x_test_pred_optuna) ** 2, axis=1)
print(recon_errors_optuna_bez[:10])

plt.figure(figsize=(8, 5))
plt.hist(recon_errors_optuna_bez, bins=40, edgecolor='black')
plt.title("Histogram błędów rekonstrukcji – Optuna bez pruning")
plt.xlabel("Reconstruction error (MSE)")
plt.ylabel("Liczba próbek")
plt.grid(True)
plt.show()


## Ray tune z ASHA

In [ ]:
import time
import psutil
import numpy as np

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.metrics import SparseCategoricalAccuracy

from ray import tune
from ray.air import session
from ray.tune import Tuner
from ray.tune.tune_config import TuneConfig
from ray.tune.schedulers import ASHAScheduler

input_dim = x_train_scaled.shape[1]
num_classes = len(np.unique(y_train))


def build_classifier_model(config, input_dim, num_classes):
    units_1      = config["units_1"]
    units_2      = config["units_2"]
    encoding_dim = config["encoding_dim"]
    class_units  = config["class_units"]
    lr           = config["lr"]

    inputs = layers.Input(shape=(input_dim,))

    # encoder
    x = layers.Dense(units_1, activation="relu")(inputs)
    x = layers.Dense(units_2, activation="relu")(x)
    bottleneck = layers.Dense(encoding_dim, activation="relu", name="bottleneck")(x)

    # classifier
    c = layers.Dense(class_units, activation="relu")(bottleneck)
    class_out = layers.Dense(num_classes, activation="softmax", name="class_output")(c)

    model = keras.Model(inputs=inputs, outputs=class_out)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=[SparseCategoricalAccuracy(name="accuracy")]
    )

    return model


def train_autoencoder_supervised_clf(config, x_train, y_train, x_val, y_val, epochs=50, batch_size=64):
    model = build_classifier_model(
        config,
        input_dim=x_train.shape[1],
        num_classes=len(np.unique(y_train))
    )

    es = keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=10,
        mode="max",
        restore_best_weights=True
    )

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=[es]
    )

    best_val_acc = float(np.nanmax(history.history["val_accuracy"]))
    session.report({"val_accuracy": best_val_acc})


# Przestrzeń hiperparametrów
search_space = {
    "units_1": tune.choice([16, 32, 64, 128]),
    "units_2": tune.choice([8, 16, 32, 64]),
    "encoding_dim": tune.choice([4, 6, 8, 10, 12]),
    "class_units": tune.choice([8, 16, 32, 64]),
    "lr": tune.choice([0.001, 0.003, 0.005, 0.01]),
}

# Uruchomienie Tuner-a (z harmonogramem ASHA)
start = time.perf_counter()
process = psutil.Process()

scheduler = ASHAScheduler(
    max_t=50,
    grace_period=5,
    reduction_factor=3
)

tuner_ray = Tuner(
    tune.with_parameters(
        train_autoencoder_supervised_clf,
        x_train=x_train_scaled,
        y_train=y_train,
        x_val=x_val_scaled,
        y_val=y_val,
        epochs=50,
        batch_size=64,
    ),
    param_space=search_space,
    tune_config=TuneConfig(
        metric="val_accuracy",
        mode="max",
        num_samples=20,
        scheduler=scheduler,
    ),
)

results = tuner_ray.fit()

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024 * 1024)

best = results.get_best_result(metric="val_accuracy", mode="max")

print(f"\nCzas tuningu: {elapsed:.2f} s")
print(f"Zużycie pamięci: {mem_used:.2f} MB")
print("Najlepszy config:", best.config)
print("Najlepsza val_accuracy:", best.metrics["val_accuracy"])


def build_autoencoder(config, input_dim, num_classes):
    units_1 = config["units_1"]
    units_2 = config["units_2"]
    encoding_dim = config["encoding_dim"]
    class_units = config["class_units"]
    lr = config["lr"]

    inp = layers.Input(shape=(input_dim,))

    # ENCODER
    x = layers.Dense(units_1, activation="relu")(inp)
    x = layers.Dense(units_2, activation="relu")(x)
    bottleneck = layers.Dense(encoding_dim, activation="relu", name="bottleneck")(x)

    # DECODER
    d = layers.Dense(units_2, activation="relu")(bottleneck)
    d = layers.Dense(units_1, activation="relu")(d)
    reconstruction = layers.Dense(
        input_dim,
        activation="linear",
        name="reconstruction"
    )(d)

    # KLASYFIKATOR
    c = layers.Dense(class_units, activation="relu")(bottleneck)
    class_out = layers.Dense(
        num_classes,
        activation="softmax",
        name="class_output"
    )(c)

    model = keras.Model(inputs=inp, outputs=[class_out, reconstruction])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss={
            "class_output": "sparse_categorical_crossentropy",
            "reconstruction": "mse"
        },
        loss_weights={
            "class_output": 1.0,
            "reconstruction": 0.5
        },
        metrics={
            "class_output": SparseCategoricalAccuracy(name="accuracy")
        }
    )

    return model


# Trening finalnego modelu (clf + rekonstrukcja)
best_model_ray = build_autoencoder(
    best.config,
    input_dim=x_train_scaled.shape[1],
    num_classes=num_classes
)

history_best_ray = best_model_ray.fit(
    x_train_scaled,
    {"class_output": y_train, "reconstruction": x_train_scaled},
    validation_data=(
        x_val_scaled,
        {"class_output": y_val, "reconstruction": x_val_scaled}
    ),
    epochs=50,
    batch_size=64,
    verbose=1,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_class_output_accuracy",
            patience=10,
            mode="max",
            restore_best_weights=True
        )
    ]
)

eval_results = best_model_ray.evaluate(
    x_test_scaled,
    {"class_output": y_test, "reconstruction": x_test_scaled},
    return_dict=True
)

print("\nWyniki na zbiorze testowym (pełny autoenkoder):")
for k, v in eval_results.items():
    print(f"{k}: {v:.4f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_best_ray.history["class_output_accuracy"], label="Train accuracy")
plt.plot(history_best_ray.history["val_class_output_accuracy"], label="Validation accuracy")
plt.xlabel("Epoka")
plt.ylabel("Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

class_pred, x_test_recon = best_model_ray.predict(x_test_scaled)
recon_errors_ray_pruning = np.mean((x_test_scaled - x_test_recon) ** 2, axis=1)

plt.figure(figsize=(8, 5))
plt.hist(recon_errors, bins=40, edgecolor="black")
plt.xlabel("Błąd rekonstrukcji (MSE)")
plt.ylabel("Liczba próbek")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Ray Tune bez ASHA

In [ ]:
import time
import psutil
import numpy as np

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.metrics import SparseCategoricalAccuracy

from ray import tune
from ray.air import session
from ray.tune import Tuner
from ray.tune.tune_config import TuneConfig

input_dim = x_train_scaled.shape[1]
num_classes = len(np.unique(y_train))

def build_classifier_model(config, input_dim, num_classes):
    units_1 = config["units_1"]
    units_2 = config["units_2"]
    encoding_dim = config["encoding_dim"]
    class_units = config["class_units"]
    lr = config["lr"]

    inputs = layers.Input(shape=(input_dim,))

    # encoder
    x = layers.Dense(units_1, activation="relu")(inputs)
    x = layers.Dense(units_2, activation="relu")(x)
    bottleneck = layers.Dense(encoding_dim, activation="relu", name="bottleneck")(x)

    # klasyfikator
    c = layers.Dense(class_units, activation="relu")(bottleneck)
    class_out = layers.Dense(num_classes, activation="softmax", name="class_output")(c)

    model = keras.Model(inputs=inputs, outputs=class_out)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=[SparseCategoricalAccuracy(name="accuracy")]
    )

    return model


def train_autoencoder_supervised_clf(config, x_train, y_train, x_val, y_val, epochs=50, batch_size=64):
    model = build_classifier_model(
        config,
        input_dim=x_train.shape[1],
        num_classes=len(np.unique(y_train))
    )

    es = keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=10,
        mode="max",
        restore_best_weights=True
    )

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=[es]
    )

    best_val_acc = float(np.nanmax(history.history["val_accuracy"]))
    session.report({"val_accuracy": best_val_acc})


# Przestrzeń hiperparametrów
search_space = {
    "units_1": tune.choice([16, 32, 64, 128]),
    "units_2": tune.choice([8, 16, 32, 64]),
    "encoding_dim": tune.choice([4, 6, 8, 10, 12]),
    "class_units": tune.choice([8, 16, 32, 64]),
    "lr": tune.choice([1e-3, 5e-4, 1e-4]),
}

start = time.perf_counter()
process = psutil.Process()

tuner_ray = Tuner(
    tune.with_parameters(
        train_autoencoder_supervised_clf,
        x_train=x_train_scaled,
        y_train=y_train,
        x_val=x_val_scaled,
        y_val=y_val,
        epochs=50,
        batch_size=64,
    ),
    param_space=search_space,
    tune_config=TuneConfig(
        metric="val_accuracy",
        mode="max",
        num_samples=20,
    ),
)

results = tuner_ray.fit()

elapsed = time.perf_counter() - start
mem_used = process.memory_info().rss / (1024 * 1024)

best = results.get_best_result(metric="val_accuracy", mode="max")

print(f"\nCzas tuningu: {elapsed:.2f} s")
print(f"Zużycie pamięci: {mem_used:.2f} MB")
print("Najlepszy config:", best.config)
print("Najlepsza val_accuracy:", best.metrics["val_accuracy"])


def build_full_supervised_autoencoder(config, input_dim, num_classes):
    units_1 = config["units_1"]
    units_2 = config["units_2"]
    encoding_dim = config["encoding_dim"]
    class_units = config["class_units"]
    lr = config["lr"]

    inp = layers.Input(shape=(input_dim,))

    # ENCODER
    x = layers.Dense(units_1, activation="relu")(inp)
    x = layers.Dense(units_2, activation="relu")(x)
    bottleneck = layers.Dense(encoding_dim, activation="relu", name="bottleneck")(x)

    # DECODER
    d = layers.Dense(units_2, activation="relu")(bottleneck)
    d = layers.Dense(units_1, activation="relu")(d)
    reconstruction = layers.Dense(
        input_dim,
        activation="linear",
        name="reconstruction"
    )(d)

    # KLASYFIKATOR
    c = layers.Dense(class_units, activation="relu")(bottleneck)
    class_out = layers.Dense(
        num_classes,
        activation="softmax",
        name="class_output"
    )(c)

    model = keras.Model(inputs=inp, outputs=[class_out, reconstruction])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss={
            "class_output": "sparse_categorical_crossentropy",
            "reconstruction": "mse"
        },
        loss_weights={
            "class_output": 1.0,
            "reconstruction": 0.5
        },
        metrics={
            "class_output": SparseCategoricalAccuracy(name="accuracy")
        }
    )

    return model


best_model_ray_no_pruning = build_full_supervised_autoencoder(
    best.config,
    input_dim=x_train_scaled.shape[1],
    num_classes=num_classes
)

history_best_ray_no = best_model_ray_no_pruning.fit(
    x_train_scaled,
    {"class_output": y_train, "reconstruction": x_train_scaled},
    validation_data=(
        x_val_scaled,
        {"class_output": y_val, "reconstruction": x_val_scaled}
    ),
    epochs=50,
    batch_size=64,
    verbose=1,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_class_output_accuracy",
            patience=10,
            mode="max",
            restore_best_weights=True
        )
    ]
)

eval_results_ray_no = best_model_ray_no_pruning.evaluate(
    x_test_scaled,
    {"class_output": y_test, "reconstruction": x_test_scaled},
    return_dict=True
)

print("\nWyniki na zbiorze testowym (pełny autoenkoder):")
for k, v in eval_results_ray_no.items():
    print(f"{k}: {v:.4f}")


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_best_ray_no.history["class_output_accuracy"], label="Train accuracy")
plt.plot(history_best_ray_no.history["val_class_output_accuracy"], label="Validation accuracy")
plt.xlabel("Epoka")
plt.ylabel("Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

class_pred, x_test_recon = best_model_ray_no_pruning.predict(x_test_scaled)
recon_errors_ray_no = np.mean((x_test_scaled - x_test_recon) ** 2, axis=1)

plt.figure(figsize=(8, 5))
plt.hist(recon_errors_ray_no, bins=40, edgecolor="black")
plt.xlabel("Błąd rekonstrukcji (MSE)")
plt.ylabel("Liczba próbek")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from scipy.stats import gaussian_kde

all_errors = np.concatenate([
    recon_errors_rs,
    recon_errors_hyperband,
    recon_errors_optuna_pruning,
    recon_errors_optuna_bez,
    recon_errors_ray_no,
    recon_errors_ray_pruning
])

x_grid = np.linspace(all_errors.min(), all_errors.max(), 300)

plt.figure(figsize=(8, 5))

for errors, label in [
    (recon_errors_rs,             "Random Search"),
    (recon_errors_hyperband,      "Hyperband"),
    (recon_errors_optuna_pruning, "Optuna + pruning"),
    (recon_errors_optuna_bez,     "Optuna bez pruning"),
    (recon_errors_ray_no,         "Ray Tune bez pruning"),
    (recon_errors_ray_pruning,    "Ray Tune + pruning"),
]:
    kde = gaussian_kde(errors)
    plt.plot(x_grid, kde(x_grid), label=label)

plt.xlabel("Reconstruction error (MSE)")
plt.ylabel("Gęstość")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
